# Time-Series Feature Engineering: AAPL

Download prices, build trading features (returns, SMAs, Bollinger Bands, RSI, volatility, next-day direction target), visualize them with dark-themed Plotly, and save the feature files. **All helpers live in `func_lib.py`**, so this notebook only sets parameters and calls functions.

| func_lib helper | Purpose |
|---|---|
| `download_ohlcv` | yfinance download, MultiIndex cleanup, `auto_adjust=True` |
| `build_ts_features` | Every feature in one pass, warm-up rows and last row dropped |
| `wilder_rsi` | Wilder RSI, identical to `ta.momentum.RSIIndicator`, so `ta` is no longer needed |
| `line_figure`, `candlestick_volume_figure`, `moving_average_figure`, `bollinger_figure`, `indicator_panel_figure`, `animated_line_figure` | Plotly charts sharing the `quant_dark` theme |

**Changes from the original**
- Features used to be computed on a `close_p` series taken *before* rows were dropped. It worked through index alignment but was fragile. They are now built in one function.
- Log returns are vectorized: `np.log(close).diff()`.
- **Mislabeled feature:** `Volatility` is the rolling std of *price*. That is correct for Bollinger Bands, so it is kept for downstream compatibility. A correctly defined `Return_Volatility` (annualized std of daily returns) is added.
- Bollinger Bands, which were computed but never plotted, now get their own chart.
- The animation is capped at about 120 frames with fixed axes. The original was O(n²) in figure size. The figure was shown three times before; it is now shown once.
- Saves `AAPL_trading_features.csv` with the 7 columns, in order, that the RL notebook reads.

In [ ]:
"""
Imports. Adds this folder (or its parent) to the Python path so func_lib can be imported,
then imports it. Importing func_lib also switches every Plotly chart to the shared dark theme.
"""
import os
import sys

# Make func_lib importable from this folder or its parent, without duplicate sys.path entries
for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.exists(os.path.join(path, 'func_lib.py')) and path not in sys.path:
        sys.path.append(path)

import func_lib   # importing it also applies the dark Plotly theme

In [ ]:
"""
Settings for the whole notebook: the ticker and date range, the indicator windows,
and the seven columns saved for the RL trading agent. Change values here, not further down.
"""
# ---- Configuration ----
TICKER = 'AAPL'
START_DATE, END_DATE = '2020-01-01', '2024-09-01'
SMA_SHORT, SMA_LONG = 20, 50
BB_WINDOW, BB_K = 20, 2          # Bollinger Band window and width (in std devs)
RSI_WINDOW = 14
PLOTLY_RENDERER = 'notebook'     # use None for JupyterLab / VS Code defaults

# Feature subset (and column order) expected by the RL trading-agent notebook
TRADING_FEATURES = ['Close', 'SMA_20', 'SMA_50', 'Volatility', 'BB_upper', 'BB_lower', 'RSI']

## 1. Download and build features

In [ ]:
"""
Downloads daily open, high, low, close and volume for the ticker with func_lib.download_ohlcv,
then shows summary statistics so you can sanity-check the data before building features.
"""
raw = func_lib.download_ohlcv(TICKER, START_DATE, END_DATE)
raw.describe()

In [ ]:
"""
Builds every feature in one call with func_lib.build_ts_features: returns, 20/50-day moving averages,
price and return volatility, Bollinger Bands, RSI and a next-day direction target. Shows the first rows.
"""
data = func_lib.build_ts_features(raw, sma_short=SMA_SHORT, sma_long=SMA_LONG,
                                  bb_window=BB_WINDOW, bb_k=BB_K, rsi_window=RSI_WINDOW)
close = data['Close']
data.head()

## 2. Visualizations

In [ ]:
"""
Candlestick chart of daily prices with volume underneath. Look for the 2020 crash and how
volume spikes on the biggest down days.
"""
func_lib.candlestick_volume_figure(data, f'{TICKER} Price and Volume').show()

In [ ]:
"""
Resamples the daily close to weekly and month-end values and plots all three together.
Lower frequencies smooth out the day-to-day noise and make the trend easier to see.
"""
# Resample to weekly / month-end frequency (last observation in each period)
weekly_close = close.resample('W').last()
monthly_close = close.resample('ME').last()

func_lib.line_figure({'Daily': (close, {'width': 1}),
                      'Weekly': weekly_close,
                      'Monthly': (monthly_close, {'width': 2.6})},
                     f'{TICKER} Daily, Weekly, and Monthly Close').show()

In [ ]:
"""
Price with the 20-day and 50-day moving averages, shaded between them. Wide gaps mark strong
trends; frequent crossings mark choppy, sideways periods.
"""
func_lib.moving_average_figure(data, f'{TICKER} Price with Moving Averages (shaded: 20/50 SMA spread)',
                               short_col=f'SMA_{SMA_SHORT}', long_col=f'SMA_{SMA_LONG}').show()

In [ ]:
"""
Bollinger Bands: the 20-day average plus and minus two standard deviations. The band widens in
fast markets and narrows in calm ones; closes outside it flag unusually strong moves.
"""
func_lib.bollinger_figure(data, f'{TICKER} Bollinger Bands ({BB_WINDOW}-day, {BB_K} std)').show()

In [ ]:
"""
Three-panel indicator chart: annualized 20-day return volatility, the 14-day RSI with its 70/30
overbought and oversold zones, and daily trading volume.
"""
func_lib.indicator_panel_figure(data, f'{TICKER} Technical Indicators',
                                vol_window=BB_WINDOW, rsi_window=RSI_WINDOW).show()

In [ ]:
"""
Animated chart that draws the closing price through time. Press play to watch the path unfold.
"""
func_lib.animated_line_figure(
    close, f'{TICKER} Closing Prices ({close.index[0]:%Y}-{close.index[-1]:%Y})'
).show(renderer=PLOTLY_RENDERER)

## 3. Save features

In [ ]:
"""
Saves two files: every feature, and the seven-column subset the RL trading-agent notebook reads
(AAPL_trading_features.csv). Prints the date range saved.
"""
data.to_csv(f'{TICKER}_time_series_features.csv')
data[TRADING_FEATURES].to_csv(f'{TICKER}_trading_features.csv')   # input for the RL agent notebook
print(f'Saved {len(data):,} rows: {data.index[0]:%Y-%m-%d} to {data.index[-1]:%Y-%m-%d}')